# 1. Environment Setup

In [1]:
# Library Imports
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import zipfile
import warnings
import os
import networkx as nx

#warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Constants
DRIVE_MOUNTED = False
DRIVE_DIR = '/content/drive'
PROJECT_DIR = '/MyDrive/CSCE_676/Project'
DATASET_DIRS = ['/IMDbDatasets']

LOAD_FROM_PKL = True
STORE_TO_PKL = False

In [2]:
# Mount Google Drive and identify files in dataset
if not DRIVE_MOUNTED:
  drive.mount(DRIVE_DIR)
  DRIVE_MOUNTED = True
for dataset_dir in DATASET_DIRS:
  print("Dataset Directory:", dataset_dir)
  print(f"  Files: {os.listdir(DRIVE_DIR + PROJECT_DIR + dataset_dir)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Directory: /IMDbDatasets
  Files: ['title.principals.tsv.gz', 'name.basics.tsv.gz', 'title.akas.tsv.gz', 'title.basics.tsv.gz', 'title.crew.tsv.gz', 'title.episode.tsv.gz', 'title.ratings.tsv.gz']


In [3]:
# For each file in the IMDb dataset directory, create a DataFrame
imdb_dataframes = {}
pkl_path = DRIVE_DIR + PROJECT_DIR + "/imdb_dataframes.pkl"

if LOAD_FROM_PKL:
  print(f"Loading imdb_dataframes dictionary from {pkl_path}...")
  with open(pkl_path, "rb") as f:
    imdb_dataframes = pickle.load(f)
  print("Done!")
else:
  for filename in os.listdir(DRIVE_DIR + PROJECT_DIR + DATASET_DIRS[0]):
    # Skip the "Also-Known-As" files. Very large and unsure of how to even meaninfully incorporate
    # into data exploration (at least at the moment)
    if filename == "title.akas.tsv.gz":
      continue

    print(f"Parsing file '{DRIVE_DIR + PROJECT_DIR + DATASET_DIRS[0] + '/' + filename};")
    imdb_dataframes[filename.split('.')[0] + "_" + filename.split('.')[1]] = pd.read_csv(DRIVE_DIR + PROJECT_DIR + DATASET_DIRS[0] + '/' + filename, compression='gzip', sep="\t")

# Save the imdb_dataframes dictionary to a pickle file
if STORE_TO_PKL:
  print(f"Saving imdb_dataframes dictionary to {pkl_path}...")
  with open(pkl_path, "wb") as f:
    pickle.dump(imdb_dataframes, f)
  print("Done!")


Loading imdb_dataframes dictionary from /content/drive/MyDrive/CSCE_676/Project/imdb_dataframes.pkl...
Done!


---

# 2. Project Scope

- **Key Takeaways from Project Milestone 1:**
  - Title data can be categorized into TV or Movie classifications (of which about 75% fall into the "TV" category), with each having a unique split of titles in a range of genres.
  - There are a wide away of professions represented in the dataset. Though the majority fall into the expected categories of "actor", "actress", "producer", "writer", and "director", other professions such as "cinematographer", " camera department", and even just "miscellaneous" are represented.
- **Dataset:**
  - [IMDb Non-Commercial Datasets](https://developer.imdb.com/non-commercial-datasets/)
- **Course Techniques:**
  - Frequent Itemsets
  - PageRank
- **External Techniques:**
  - Link Prediction

---


# 3. Research Question Definition

## RQ1: Can PageRank be used as a predictor for actor/director success?

- **Data Mining Task(s):**
  - Graph Construction
  - Time-Series Analysis
- **Relevant Algorithm(s):**
  - Standard PageRank (course)
  - Weighted/Temporal PageRank (external, optional)
- **Evaluation Criteria:**
  - Correlation Coefficient (PageRank vs. ratings)

## RQ2: Which actor combinations are associated with highly rated films?

- **Data Mining Task(s):**
  - Frequent Itemset Mining / Association Rule Mining
  - Comparative Pattern Analysis
      -  Compare frequent itemsets across different genres
- **Relevant Algorithm(s):**
  - Apriori (course)
- **Evaluation Criteria:**
  - Support
    - How does varying minsup affect generated itemsets?
  - Confidence
  - Lift

## RQ3: Which directors are likely to hire the same crew members (writes, cinematographers, etc.) again?

- **Data Mining Task(s):**
  - Link Prediction
  - Temporal Graph Construction
    - For generating graphs used for testing and training of link prediction
- **Relevant Algorithm(s):**
  - Link Prediction Algorithms (external)
    - Common Neighbors
    - Jaccard Coefficient
- **Evaluation Criteria:**
  - ROC-AUC
    - For evaluating quality of predicted links
  - F1 Score
    - For capturing an overall understanding of precision and recall

---

 # 4. Motivation and Feasibility

## RQ1: Can PageRank be used as a predictor for actor/director success?

In [4]:
# Extract needed DataFrames
principals = imdb_dataframes["title_principals"]
basics = imdb_dataframes["title_basics"]
ratings = imdb_dataframes["title_ratings"]
names = imdb_dataframes["name_basics"]

# Filter for movies only
movies = basics[basics["titleType"] == "movie"][["tconst", "startYear"]]

# Merge principals with movies to keep only movie cast/crew
movie_principals = principals.merge(movies, on="tconst", how="inner")

# Identify actors and directors
actors = movie_principals[movie_principals["category"].isin(["actor", "actress"])]
directors = movie_principals[movie_principals["category"] == "director"]

# ---------------------------------------------------------
# 1. ACTOR COLLABORATION GRAPH
# ---------------------------------------------------------
actor_graph = nx.Graph()

# Group by movie and get all actors in each film
for tconst, group in actors.groupby("tconst"):
    actor_list = group["nconst"].tolist()
    # Add edges between all pairs of actors in the same film
    for i in range(len(actor_list)):
        for j in range(i + 1, len(actor_list)):
            a1, a2 = actor_list[i], actor_list[j]
            if actor_graph.has_edge(a1, a2):
                actor_graph[a1][a2]["weight"] += 1
            else:
                actor_graph.add_edge(a1, a2, weight=1)

print("Actor graph nodes:", actor_graph.number_of_nodes())
print("Actor graph edges:", actor_graph.number_of_edges())

Actor graph nodes: 1316976
Actor graph edges: 13739077


In [5]:
# ---------------------------------------------------------
# 2. DIRECTOR COLLABORATION GRAPH
# ---------------------------------------------------------
director_graph = nx.Graph()

# Directors who worked on the same film
for tconst, group in directors.groupby("tconst"):
    director_list = group["nconst"].tolist()
    for i in range(len(director_list)):
        for j in range(i + 1, len(director_list)):
            d1, d2 = director_list[i], director_list[j]
            if director_graph.has_edge(d1, d2):
                director_graph[d1][d2]["weight"] += 1
            else:
                director_graph.add_edge(d1, d2, weight=1)

print("Director graph nodes:", director_graph.number_of_nodes())
print("Director graph edges:", director_graph.number_of_edges())


Director graph nodes: 78038
Director graph edges: 64767


In [6]:
# ---------------------------------------------------------
# 3. ACTOR–DIRECTOR BIPARTITE GRAPH
# ---------------------------------------------------------
actor_director_graph = nx.Graph()

# Add edges between actors and directors who worked on the same film
for tconst, group in movie_principals.groupby("tconst"):
    film_actors = group[group["category"].isin(["actor", "actress"])]["nconst"].tolist()
    film_directors = group[group["category"] == "director"]["nconst"].tolist()

    for a in film_actors:
        for d in film_directors:
            if actor_director_graph.has_edge(a, d):
                actor_director_graph[a][d]["weight"] += 1
            else:
                actor_director_graph.add_edge(a, d, weight=1)

print("Actor–Director graph nodes:", actor_director_graph.number_of_nodes())
print("Actor–Director graph edges:", actor_director_graph.number_of_edges())

Actor–Director graph nodes: 1453042
Actor–Director graph edges: 3551291


In [7]:
# ---------------------------------------------------------
# 4. COMPUTE PAGERANK FOR EACH GRAPH
# ---------------------------------------------------------
actor_pagerank = nx.pagerank(actor_graph, weight="weight")
director_pagerank = nx.pagerank(director_graph, weight="weight")
actor_director_pagerank = nx.pagerank(actor_director_graph, weight="weight")

print("Computed PageRank for all graphs!")

Computed PageRank for all graphs!


In the above code, we create a graph indicating the collaborations between actors as well as a graph for indicating the collaborations between directors. Additionally, because certain actor/director pairings may be relevant for predicting actor/director success a third, bipartite graph is generated. Afterward, for each graph the PageRank is calculate, which could then be used in future regression/classification tasks.


- **Motivation:**
  - Dataset has numerous attributes, whether provided or calculable, that provide the means for performing weighted PageRank with a large number of weigthing criteria.
- **Non-triviality:**
  - There is vast amount of exploration possible for identifying how various critieria (number of appearances, size of peer network, etc.) can be factored into a weighted PageRank implementation to improve the effectiveness of predictions.
- **Feasibility:**
  - Had initial concerns with whether or not graphs would fit in available RAM (using "High-RAM" runtime in Google Colab with 50GB). Initial EDA appears to indicate that this is feasible. Though may need to delete graphs at each iteration (if performing temporal PageRank) and save only PageRank values to prevent a situation where all RAM is exhausted after only a few iterations.
- **Risks:**
  - Compute time. Graph construction of a single set of graphs (actor collaboration, director collaboration, actor-director bipartite) takes about 13 minutes (with the bulk of this time spent constructing the actor-director bipartite graph). If creating many sets of graphs either at various temporal snapshot (temporal PageRank) or of different weighting criteria (weighted PageRank), then compute time may become intractable without sampling or excluding use of the bipartite graph in analysis.

## RQ2: Which actor combinations are associated with highly rated films?

In [8]:
# ---------------------------------------------------------
# 1. HIGH-RATED ACTOR/ACTRESS EXTRACTION
# ---------------------------------------------------------
# Merge movies with ratings
movies_with_ratings = movies.merge(ratings, on="tconst", how="inner")

# Keep only highly-rated films (threshold = 8.0)
high_rated_movies = movies_with_ratings[movies_with_ratings["averageRating"] >= 8.0]

# Merge with principals to get actors in those films
high_rated_principals = principals.merge(high_rated_movies, on="tconst", how="inner")

# Keep only actors/actresses
high_rated_actors = high_rated_principals[
    high_rated_principals["category"].isin(["actor", "actress"])
][["tconst", "nconst"]]

print(f"Num. high-rated movies: {len(high_rated_movies)}")
print(f"Num. high-rated principals: {len(high_rated_principals)}")
print(f"Num. high-rated actors: {len(high_rated_actors)}")


Num. high-rated movies: 29570
Num. high-rated principals: 382593
Num. high-rated actors: 146540


In [9]:
# ---------------------------------------------------------
# 2. TRANSACTION GENERATION
# ---------------------------------------------------------
# Group actors by movie. Each film is mapped to a list of
# actors
transactions = (
    high_rated_actors.groupby("tconst")["nconst"]
    .apply(list)
    .reset_index()
)

In [10]:
# ---------------------------------------------------------
# 3. ONE-HOT ENCODE TRANSACTIONS
# ---------------------------------------------------------
te = TransactionEncoder()
te_array = te.fit(transactions["nconst"]).transform(transactions["nconst"])

df_transactions = pd.DataFrame(te_array, columns=te.columns_)

In [11]:
# ---------------------------------------------------------
# 4. APRIORI APPLICATION FOR FREQ. ITEMSET DETECTION
# ---------------------------------------------------------
# Minimum support = appear in at least 0.05% of high-rated films
frequent_itemsets = apriori(
    df_transactions,
    min_support=0.0005,
    use_colnames=True
)

frequent_itemsets["itemset_size"] = frequent_itemsets["itemsets"].apply(lambda x: len(x))
frequent_itemsets.sort_values("support", ascending=False).head()

print(f"Num. Frequent Itemsets Identified: {len(frequent_itemsets)}")
for sz, cnt in frequent_itemsets["itemset_size"].value_counts().items():
  print(f"  {sz}-Itemsets: {cnt}")

Num. Frequent Itemsets Identified: 642
  1-Itemsets: 407
  2-Itemsets: 155
  3-Itemsets: 76
  4-Itemsets: 4


In the above code, we extract highly-rated movies (those with a rating at or above 8.0), identify the actors/actresses associated with them, and convert those lists into transaction. We then one-hot encode our transactions. Lastly we use the same Apriori library as had been used in prior homework to identify our frequent itemsets. With the minsup that was used for this initial, feasibility test, a minsup of 0.05% yielded 642 frequent itemsets. Though, as it applies to the original research question, the frequent 1-itemsets may not be of that much interest, which would leave only 235 other frequent itemsets.

- **Motivation:**
  - Interesting area of study. Can be used to quantifiably identify "chemistry" between actors/actresses that result in well-received movies.
- **Non-triviality:**
  - Room for utilizing dynamic minsup based on career duration (would require use of an algorithm like Multiple Minimum Support - i.e. MIS) or based on genre, as well as tuning based on definition of "highly-rated film". As above EDA show, minsup tuning would be critical for identifying meaningful conclusions.
- **Feasibility:**
  - Computationally very feasible (both in time and space). EDA performed above took only a few minutes.  Unsure how much this changes if MIS is utilized.
- **Risks:**
  - Lack of valuable results if parameters are not tuned effectively. Even with properly tuned parameters, are the insights arrived at that meaningful?

## RQ3: Which directors are likely to hire the same crew members (writes, cinematographers, etc.) again?

In [12]:
# ---------------------------------------------------------
# 1. EXTRACT DIRECTORS AND CREW
# ---------------------------------------------------------
directors = movie_principals[movie_principals["category"] == "director"][["tconst", "nconst"]]
crew = movie_principals[movie_principals["category"].isin([
    "writer", "cinematographer", "editor", "composer", "producer"
])][["tconst", "nconst"]]

directors = directors.rename(columns={"nconst": "director"})
crew = crew.rename(columns={"nconst": "crew"})

In [13]:
# ---------------------------------------------------------
# 2. GENERATE DIRECTOR-CREW PAIRS
# ---------------------------------------------------------

director_crew_pairs = directors.merge(crew, on="tconst", how="inner")

# Remove pairs where the director also had another role (i.e. remove
# self-referential pairs)
director_crew_pairs = director_crew_pairs[director_crew_pairs["director"] != director_crew_pairs["crew"]]
pair_counts = (
    director_crew_pairs.groupby(["director", "crew"])
    .size()
    .reset_index(name="num_collaborations")
)

# Sort by most repeated collaborations
pair_counts_sorted = pair_counts.sort_values("num_collaborations", ascending=False)
print(f"Num. unique director-crew pairs: {len(pair_counts)}")

# Map director/crew nconsts to actual names
nconst_to_name = names.set_index("nconst")["primaryName"].to_dict()
def map_itemset_to_names(itemset):
    return [nconst_to_name[n] for n in itemset]

pair_counts_sorted["director_name"] = pair_counts_sorted["director"].map(nconst_to_name)
pair_counts_sorted["crew_name"] = pair_counts_sorted["crew"].map(nconst_to_name)

col = pair_counts_sorted.pop("crew_name")
pair_counts_sorted.insert(0, "crew_name", col)

col = pair_counts_sorted.pop("director_name")
pair_counts_sorted.insert(0, "director_name", col)

display(pair_counts_sorted.drop(columns=["director", "crew"], axis=1).head())

Num. unique director-crew pairs: 1925082


,director_name,crew_name,num_collaborations
1073442,Sachi Hamano,Kuninori Yamazaki,155
980940,Nick Randall,Jordan Hill,143
838374,Minoru Inao,Shôji Sakai,135
559983,Sam Newfield,Sigmund Neufeld,125
421842,Satoru Kobayashi,Tomoki Yanagida,122


The code above extracts directors as well as othe non-actor/non-actress roles from the principals dataset. Afterward these pairings are mapped to each other based on occurences in which a director worked with a specific crew member. We make sure to remove self-referential paris that would distort our analysis. Lastly, we map `nconsts` back to actual names and print the results for visual inspection (which is how I identified that self-referential pairs needed to be filtered out).

- **Motivation:** Extracting interactions between director and crew enables us to understand the nature of filmmaking as a joint effort. Studying these recurring partnerships can reveal which directors cultivate long‑term creative teams, whether certain collaborations are associated with higher critical success. We may want to merge aspects of RQ2 with RQ3 (i.e. don't just identify collaboration - act on them too).
- **Non-triviality:** To add additional depth to this question, I would need want to expand upon the scope of the question - e.g. what "unusual" collaborations exist (i.e. director-crew pairs where both typically work in other geos or genres).
- **Feasibility:** Link prediction algorithms appear to be computationally expensive, both in time and space. RQ3 portion of Part 5 (below) caused runtime to exhaust available RAM while attempting to calculate Jaccard coefficients.
- **Risks:** Without merging with aspects of other questions (i.e. RQ2), I;m unsure if the scope of this question is particularly interesting. Collaboration identification alone, without additional exploration (see RQ3 section of Part 3), may lead to shallow analysis.

---

# 5. Methodological Planning

## RQ1: Can PageRank be used as a predictor for actor/director success?

In [14]:
# ---------------------------------------------------------
# 1. COMPUTE ACTOR RATINGS
# ---------------------------------------------------------
# Merge actors with ratings
actor_ratings = actors.merge(ratings, on="tconst", how="inner")

# Compute average rating per actor
actor_avg_rating = (
    actor_ratings.groupby("nconst")["averageRating"]
    .mean()
    .reset_index()
    .rename(columns={"averageRating": "avg_rating"})
)

# Convert PageRank dict to DataFrame
actor_pr_df = (
    pd.DataFrame.from_dict(actor_pagerank, orient="index", columns=["pagerank"])
    .reset_index()
    .rename(columns={"index": "nconst"})
)

# Merge PageRank with average rating
actor_regression_df = actor_pr_df.merge(actor_avg_rating, on="nconst", how="inner")

In [15]:
# ---------------------------------------------------------
# 2. FIT AND TRAIN REGRESSION MODEL
# ---------------------------------------------------------
# Features and target
X = actor_regression_df[["pagerank"]]
y = actor_regression_df["avg_rating"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
reg = LinearRegression()
reg.fit(X_train, y_train)

# Predict
y_pred = reg.predict(X_test)


In [16]:
# ---------------------------------------------------------
# 3. EVALUATE PREDICTIVE ACCURACY
# ---------------------------------------------------------
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)


MSE: 1.8173673193835826
R²: 2.7040075466500113e-06


From the `actor_pagerank` scores calculate for RQ1 in Part 4, we train a regression model and train with a subset of the actor data. Interestingly, the initial findings seem to indicate that standard PageRank of the actor collaboration graph is not effective at all (i.e. *very* low R²) when used to predict actor success (as measured by average rating of all films the actor appeared in). If further exploration of this question is performed, then we would want to explore calculating weighted PageRank values according to different criteria to see if model predictions can be improved. Additionally, we might explore if PageRank values are better for calculating other metrics related to the actor/director careers.

- **Course Algorithms:**
  - PageRank
- **External Algorithms:**
  - Weighted/Temporal PageRank
- **Evaluation:**
  - Utilize PageRank in either regression or classification (e.g. successful/unsuccessful) modeling and map to ground truth. Could explore if models are suitable for predicting various ground truths (i.e. average rating, career longevity, etc.).
- **Baselines:**
  - Use accuracy of standard PageRank's prediction of actor/director success as the control when evaluating alternative approaches (e.g. weighted PageRank).

## RQ2: Which actor combinations are associated with highly rated films?

In [17]:
# ---------------------------------------------------------
# 1. COMPUTE RULES FROM FREQ. ITEMSETS
# ---------------------------------------------------------
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.1
)

# Map actor nconsts to actual actor names
nconst_to_name = names.set_index("nconst")["primaryName"].to_dict()
def map_itemset_to_names(itemset):
    return [nconst_to_name[n] for n in itemset]

rules["antecedent_names"] = rules["antecedents"].apply(map_itemset_to_names)
rules["consequent_names"] = rules["consequents"].apply(map_itemset_to_names)

col = rules.pop("consequent_names")
rules.insert(0, "consequent_names", col)

col = rules.pop("antecedent_names")
rules.insert(0, "antecedent_names", col)


In [18]:
# ---------------------------------------------------------
# 2. SORT RULES ACCORDING TO CONFIDENCE AND LIFT
# ---------------------------------------------------------
print("Highest-confidence actor combinations")
display(rules.sort_values("confidence", ascending=False).drop(columns=["antecedents", "consequents"], axis=1).head())

print("\nHighest-lift actor combinations")
display(rules.sort_values("lift", ascending=False).drop(columns=["antecedents", "consequents"], axis=1).head())


Highest-confidence actor combinations


,antecedent_names,consequent_names,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
717,"[Luis Huizar, Jesus Heredia, Jesus Esparza]",[Vianey Huizar],0.00055,0.002500,0.00055,1.0,400.060000,1.0,0.000549,inf,0.998049,0.220000,1.0,0.610000
707,"[Jesus Heredia, Rafael Solis]","[Vianey Huizar, Luis Huizar]",0.00070,0.002450,0.00070,1.0,408.224490,1.0,0.000698,inf,0.998249,0.285714,1.0,0.642857
38,[Nagappa],[Rajkumar],0.00080,0.009699,0.00080,1.0,103.108247,1.0,0.000792,inf,0.991094,0.082474,1.0,0.541237
50,[Sampath],[Rajkumar],0.00060,0.009699,0.00060,1.0,103.108247,1.0,0.000594,inf,0.990896,0.061856,1.0,0.530928
700,"[Vianey Huizar, Jesus Heredia, Rafael Solis]",[Luis Huizar],0.00070,0.002450,0.00070,1.0,408.224490,1.0,0.000698,inf,0.998249,0.285714,1.0,0.642857



Highest-lift actor combinations


,antecedent_names,consequent_names,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
282,[Babymetal],[Suzuka Nakamoto],0.00060,0.00065,0.0006,1.000000,1538.692308,1.0,0.0006,inf,0.99995,0.923077,1.000000,0.961538
283,[Suzuka Nakamoto],[Babymetal],0.00065,0.00060,0.0006,0.923077,1538.692308,1.0,0.0006,12.992201,1.00000,0.923077,0.923031,0.961538
285,[Babymetal],[Moa Kikuchi],0.00060,0.00065,0.0006,1.000000,1538.692308,1.0,0.0006,inf,0.99995,0.923077,1.000000,0.961538
284,[Moa Kikuchi],[Babymetal],0.00065,0.00060,0.0006,0.923077,1538.692308,1.0,0.0006,12.992201,1.00000,0.923077,0.923031,0.961538
684,[Moa Kikuchi],"[Babymetal, Suzuka Nakamoto]",0.00065,0.00060,0.0006,0.923077,1538.692308,1.0,0.0006,12.992201,1.00000,0.923077,0.923031,0.961538


In the above code, we identify the association rules from the itemsets generated in RQ2 of Part 4. The interpretation of the rules is essentially "in highly-rated films featuring `{antecedent actors}`, `{consequent actors}` appear as well." For the rules with the highest lift, we might conclude that actors in the antecedent–consequent pairs (or groups) co‑appear in highly‑rated films far more often than would be expected by chance, indicating unusually strong, meaningful collaboration patterns.

- **Course Algorithms:**
  - Apriori
- **External Algorithms:**
  - Multiple Minimum Support (MIS)
    - Evaluate frequent itemsets and rules generated based on using dynamic minsups based on various principal criteria (career duration, appearances, etc.).
- **Evaluation:**
  - Utilize lift to determine whether or not the identified rules are meaningful (i.e. not simply due to chance)
- **Baselines:**
  - High-support, static, frequent itemsets

## RQ3: Which directors are likely to hire the same crew members (writes, cinematographers, etc.) again?

In [19]:
# ---------------------------------------------------------
# 1. DIRECTOR-CREW BIPARTITE GRAPH CONSTRUCTION
# ---------------------------------------------------------
G = nx.Graph()

# Add weighted edges for each director–crew pair
for _, row in pair_counts.iterrows():
    director = row["director"]
    crew = row["crew"]
    weight = row["num_collaborations"]

    G.add_edge(director, crew, weight=weight)

print("Director-Crew graph nodes:", G.number_of_nodes())
print("Director-Crew graph edges:", G.number_of_edges())

Director-Crew graph nodes: 897634
Director-Crew graph edges: 1890218


In [20]:
# ---------------------------------------------------------
# 2. IMPLEMENT LINK PREDICTION (VIA JACCARD COEFFICIENT)
# ---------------------------------------------------------
# Reduce set of pairs for which we calculate the
# Jaccard cofficient to those that share at least one neighbor
# Otherwise scope of calculation becomes intractable for
# the runtime.
candidate_pairs = set()
for node in G.nodes():
    neighbors = list(G.neighbors(node))
    for i in range(len(neighbors)):
        for j in range(i+1, len(neighbors)):
            candidate_pairs.add((neighbors[i], neighbors[j]))

# Calculate Jaccard coefficients
jaccard_scores = nx.jaccard_coefficient(G, candidate_pairs)

predictions = [
    (u, v, score)
    for u, v, score in jaccard_scores
    if not G.has_edge(u, v)
]

predictions_sorted = sorted(predictions, key=lambda x: x[2], reverse=True)

In [22]:
print(f"Num. Candidate Pairs: {len(candidate_pairs)}")

pred_df = pd.DataFrame(predictions_sorted, columns=["director", "crew", "score"])

# Map director/crew nconsts to actual names
nconst_to_name = names.set_index("nconst")["primaryName"].to_dict()
def map_itemset_to_names(itemset):
    return [nconst_to_name[n] for n in itemset]

pred_df["director_name"] = pred_df["director"].map(nconst_to_name)
pred_df["crew_name"] = pred_df["crew"].map(nconst_to_name)

col = pred_df.pop("crew_name")
pred_df.insert(0, "crew_name", col)

col = pred_df.pop("director_name")
pred_df.insert(0, "director_name", col)

pred_df[["director_name", "crew_name", "score"]].head()

Num. Candidate Pairs: 34126591


,director_name,crew_name,score
0,Ricardo Mistral,Poldy Bird,1.0
1,Chuck Brown,Ryan Lamy,1.0
2,George Chu,Daryl Forst,1.0
3,Kaare Hersoug,Gerd Fleischer,1.0
4,Adv Gayathri Nair,Aju Thomas,1.0


In the above code we create a graph that shows the captures the interactions between the directors and various crew members in our dataset. To make the Jaccard coefficient calculation feasible for the runtimes RAM limitations (50GB), we calculate only for a subset of the node pairs that exist (those with at least one common neighbor). From our initial exploration we then show the highest-predicted director-crew pairs that do not currently exist in the graph.

For the Jaccard cofficients calculated we can see that the highest were 1.0. This means that the director and crew member in each pair share identical neighborhoods in the graph (i.e. they have worked with exactly the same set of people, and no one else). If future/additional analysis is performed, then we may want to filter nodes based on a minimum number of interactions.


- **Course Algorithms:**
  - F1 Scoring (for evaluation)
- **External Algorithms:**
  - Link Prediction
    - Preferential Attachment
    - Adamic-Adar Index
    - Jaccard Coefficient
- **Evaluation:**
  - Identify which prediction algorithm yields the most accurate predictions when applied to time-sliced/parital constructions of the director-crew bipartite graph.
- **Baselines:**
  - Preferential Attachment

---

# 6. Collaboration Declaration

- **Collaborators:** N/A
- **Web Sources:**
  - IMDb Non-Commercial Datasets: https://developer.imdb.com/non-commercial-datasets/
- **AI Tools:**
  - Microsoft Copilot: General assistance with coding of EDA (e.g. graph construction and regression model training for RQ1, transaction encoding for RQ2, link prediction for RQ3, etc.)
- **Citations:** N/A